In [7]:
import re
import os
import pandas as pd
import numpy as np
from pathlib import PurePath, PurePosixPath

import sys
sys.path.append('/disk1/users/da_cyh_0/PROJECT/DATA/anony')
from h_anonypy.modules_dicom import get_folder_list, replace_digits, check_dicom_from_folder
from h_anonypy.modules_dicom import get_patient_info, anonymize_dicom_file

from pydicom.errors import InvalidDicomError
def check_dcm_from_folder(root_dir, return_list=False):
    

    dcm_dir_list = []
    for dirpath, _, filenames in os.walk(root_dir):
        for file in filenames:
            if file != "DICOMDIR" and not file.startswith('._'):
                filepath = os.path.join(dirpath, file)
                try:
                    pydicom.dcmread(filepath, stop_before_pixels=True)
                    if return_list == True:
                        dcm_dir_list.append(dirpath)
                        break
                    else:
                        return filepath
                except InvalidDicomError:
                    continue
                except Exception:
                    continue
    if return_list == True:
        return dcm_dir_list
    else:
        return False

def get_meta_info(dicom_file):
    ds = pydicom.dcmread(dicom_file)
    ds.SpecificCharacterSet = 'ISO_IR 192'  # UTF-8
    # ds.SpecificCharacterSet = 'ISO_IR 149'  # EUC-KR
    ds.decode()
    info = {
        'PatientID':    ds.get('PatientID'),
        'PatientName':  str(ds.get('PatientName')),
        'PatientSex':   ds.get('PatientSex'),
        'PatientAge':   ds.get('PatientAge'),
        'PatientBirthDate': ds.get('PatientBirthDate'),
        'AcquisitionDate': ds.get('AcquisitionDate', 'No AcquisitionDate'),
        'PatientSize': ds.get('PatientSize'),
        'PatientWeight': ds.get('PatientWeight'),
        'OtherPatientIDs': ds.get('OtherPatientIDs'),
        'OtherPatientNames': str(ds.get('OtherPatientNames')),
        'InstitutionName': ds.get('InstitutionName'),
        'ReferringPhysicianName': str(ds.get('ReferringPhysicianName')),
        'AccessionNumber': ds.get('AccessionNumber'),
        'Modality': ds.get('Modality'),
        'BodyPartExamined': ds.get('BodyPartExamined'),
        'Manufacturer': str(ds.get('Manufacturer')),
        'ManufacturerModelName': str(ds.get('ManufacturerModelName'))

    }
    return info



In [ ]:
# check spacing for nift files

from tqdm.notebook import tqdm
import sys
from pathlib import Path

sys.path.append('/home/yhchoi/PROJECT/Pneumo')
from pneumo_reconpy import (ImageReader,
                            AbdomenSegmenter, 
                            UmbilicusDetector, 
                            predictUmbilicus)

from pneumo_reconpy import (sitk_normalize_array, 
                            sitk_transform_and_resample,
                            sitk_compute_rotation)

# IMAGE_META = pd.read_excel('./SHEET/IMAGE_META.xlsx', sheet_name=None)
# VIDEO_META = pd.read_excel('./SHEET/VIDEO_META.xlsx', sheet_name=None)
# HUTOM_ID = pd.read_excel('./SHEET/HUTOM_ID.xlsx', sheet_name=None)

UGI_META = IMAGE_META['UGI'].copy()
UGI_META_CT = UGI_META[UGI_META['Modality']=='CT'].copy()
UGI_META_CT = UGI_META_CT[~(UGI_META_CT['Center']=='GradientHealth')].reset_index(drop=True)

select = UGI_META_CT[UGI_META_CT['AcquisitionDate']=='19000101'].reset_index(drop=True)
select['N.slices'] = 0
select['Thickness'] = 0

hid_list = select['hutom_id'].tolist()
base_dir = Path('/nas/nas6/UGI/CT_SurgGram')
for i in tqdm(range(len(hid_list))):

    sample_folder = base_dir / hid_list[i] / '00_DICOM'

    sample_volumes = []
    sample_thcknesses = []
    sample_paths = []
    for dirpath, _, filenames in os.walk(sample_folder):
        for filename in filenames:
            if '_iso.' not in filename:
                sample_dir = Path(dirpath) / filename
                sample_volume = ImageReader(sample_dir).sitk_volume
                sample_volumes.append(sample_volume)
                sample_thcknesses.append(sample_volume.GetSpacing()[2])
                sample_paths.append(Path(dirpath))

    # slicethickness가 가장 얇은 볼륨 선택
    idx = np.argmin(sample_thcknesses)
    sample_dir = sample_paths[idx]
    sample_volume = sample_volumes[idx]
    select.loc[i,'N.slices'] = sample_volume.GetDepth()
    select.loc[i,'Thickness'] = sample_thcknesses[idx]

select



In [10]:
##### INPUT #####
# IMAGE_META : 전체 영상 메타정보 
# HUTOM_ID : 전체 hutom id 리스트
# dicom_dir : 반입 데이터 경로
# center: 반입 기관
# importdate: 반입 날짜
# organ: 조직 정보
# n_digits: hutom id 자리수

os.listdir('/nas/nas6/DataTeam/UGI/[CT]신촌세브란스 501')

IMAGE_META = pd.read_excel('/disk1/users/da_cyh_0/PROJECT/DATA/anony/SHEET/IMAGE_META.xlsx', sheet_name=None)
HUTOM_ID = pd.read_excel('/disk1/users/da_cyh_0/PROJECT/DATA/anony/SHEET/HUTOM_ID.xlsx', sheet_name=None)
dicom_dir = ['/nas/nas6/DataTeam/UGI/[CT]신촌세브란스 501']
center = ['SEVERANCE']
importdate = ['20251226']
organ = ['UGI']
n_digits = 4


In [ ]:
## Start [ver.2025.08]

sample_info_list = []
for i in range(len(dicom_dir)):

    # 1. 입력 폴더 내 샘플 폴더 리스트
    folder_list = get_folder_list(dicom_dir[i])
    for j in range(len(folder_list)):

        # 1.1. 샘플 폴더 기준, DICOM 확인 및 메타 추출
        check_path = os.path.join(dicom_dir[i], folder_list[j])
        dcm_fname = check_dicom_from_folder(check_path)
        if dcm_fname:
            metadata = get_patient_info(dcm_fname)

            anony_folder = re.sub(r'[가-힣]+', '', folder_list[j])
            anony_folder = re.sub(r'\d+', replace_digits, anony_folder)
            anony_folder = re.sub(r'\s+', '', anony_folder)

            posix = PurePath(dcm_fname)
            fpath = PurePosixPath(organ[i], *posix.parts[2:-1])
            metadata['folder'] = fpath
            metadata['Center'] = center[i]
            metadata['ImportDate'] = importdate[i]
            
            sample_info_list.append(metadata)

    # 2. 샘플 메타 [반입리스트에서 중복 제거 ? default=False]
    sample_info = pd.DataFrame.from_dict(sample_info_list)
    check_dup_col = ['PatientID','PatientName','AcquisitionDate','PatientSex']
    # sample_info = sample_info.drop_duplicates(subset=check_dup_col, ignore_index=True)
    sample_info.insert(0, 'hutom_id', None)

    # 3. 중복 제거 및 HUTOM ID 부여
    meta_all = IMAGE_META[organ[i]].copy()
    ids_all = HUTOM_ID[organ[i]].copy()
    mask = ~ids_all['hutom_id'].astype(str).str.contains('FDA', case=False, na=False)
    hutom_ids = ids_all.loc[mask,'hutom_id'].dropna().unique().tolist()
    nums = [int(m.group(1)) for s in hutom_ids if (m := re.search(r'(\d+)$', s))]
    id_number = np.sort(nums)[-1] + 1

    sample_ids = sample_info['PatientID'].unique().tolist()
    for idx in range(len(sample_ids)):
        dup = meta_all[meta_all['PatientID']==sample_ids[idx]]
        if len(dup) == 0:
            hutomid = f"{organ[i]}{id_number:0{n_digits}d}"
            check_sample = sample_info[sample_info['PatientID'] == sample_ids[idx]]
            sample_info.loc[check_sample.index, "hutom_id"] = hutomid
            id_number += 1
        elif len(dup) > 0:
            hutomid = dup["hutom_id"].tolist()[0]
            check_sample = sample_info[sample_info['PatientID'] == sample_ids[idx]]
            sample_info.loc[check_sample.index, "hutom_id"] = hutomid
        else:
            print('check sample')

    # 4. Anonymous
    anonymous_dir = os.path.join(dicom_dir[i], 'ANONYMOUS')
    sample_ids = sample_info["PatientID"].unique().tolist()
    for idx in range(len(sample_ids)):
        # 4.1. 샘플 선택
        check_sample = sample_info[sample_info['PatientID'] == sample_ids[idx]]

        # 4.2. 폴더/경로, 영상 종류, HUTOM ID
        sample_path = check_sample["folder"].tolist()
        hutomid = check_sample['hutom_id'].tolist()    
        for m in range(len(sample_path)):
            # 샘플 기준, 하위 폴더 탐색
            posix = PurePath(sample_path[m])
            dcm_path = os.path.join(dicom_dir[i], posix.parts[2])

            dcm_folder_list = check_dicom_from_folder(dcm_path, return_list=True)
            for n in range(len(dcm_folder_list)):
                # 하위 폴더 기준, 익명화: Name, ID > Hutom ID
                raw_dir = dcm_folder_list[n]
                no_ko_folder = re.sub(r'[\u1100-\u11FF\u3130-\u318F\uAC00-\uD7A3]+', '', 
                                    posix.parts[2]).strip()

                save_dir = os.path.join(anonymous_dir, hutomid[m], 
                                        no_ko_folder, os.path.join(*posix.parts[3:]))

                os.makedirs(save_dir, exist_ok=True)

                for dirpath, _, filenames in os.walk(raw_dir):
                    for filename in filenames:
                        anonymize_dicom_file(raw_dir, filename, save_dir, id=hutomid[m])

    # 5. Add information > check !!!!
    ids_add = pd.DataFrame(sample_info['hutom_id'].unique().tolist(),
                        columns=['hutom_id'])
    ids_add['dicom'] = 'O'
    ids_all_add = pd.merge(ids_all, ids_add, on='hutom_id', how='outer')
    ids_all_add["dicom"] = ids_all_add["dicom_x"].fillna(ids_all_add["dicom_y"])
    ids_all_add = ids_all_add[['hutom_id','dicom','video']]
    meta_all_add = pd.concat([meta_all, sample_info], ignore_index=True)

##### OUTPUT > IMAGE_META, HUTOM_ID 업데이트 및 저장
# IMAGE_META[organ[i]]
# meta_all_add




In [11]:
# Start [ver.2025.12]

i = 0

# 1. 입력 폴더 내 샘플 폴더 리스트
sample_info_list = []
folder_list = get_folder_list(dicom_dir[i])
for j in range(len(folder_list)):

    # 1.1. 샘플 폴더 기준, DICOM 확인 및 메타 추출
    check_path = os.path.join(dicom_dir[i], folder_list[j])
    dcm_fname = check_dicom_from_folder(check_path)
    if dcm_fname:
        metadata = get_patient_info(dcm_fname)

        anony_folder = re.sub(r'[가-힣]+', '', folder_list[j])
        anony_folder = re.sub(r'\d+', replace_digits, anony_folder)
        anony_folder = re.sub(r'\s+', '', anony_folder)

        posix = PurePath(dcm_fname)
        # fpath = PurePosixPath(organ[i], *posix.parts[2:-1])
        fpath = PurePosixPath(*posix.parts[:-1])
        metadata['folder'] = fpath
        metadata['Center'] = center[i]
        metadata['ImportDate'] = importdate[i]
        
        sample_info_list.append(metadata)

# 2. 샘플 메타 [반입리스트에서 중복 제거 ? default=False]
sample_info = pd.DataFrame.from_dict(sample_info_list)
check_dup_col = ['PatientID','PatientName','AcquisitionDate','PatientSex']
# sample_info = sample_info.drop_duplicates(subset=check_dup_col, ignore_index=True)
sample_info.insert(0, 'hutom_id', None)
sample_info['PatientID'] = sample_info['folder'].astype(str).str.split('/').str[6]

# 3. 중복 제거 및 HUTOM ID 부여
meta_all = IMAGE_META[organ[i]].copy()
ids_all = HUTOM_ID[organ[i]].copy()
mask = ~ids_all['hutom_id'].astype(str).str.contains('FDA', case=False, na=False)
hutom_ids = ids_all.loc[mask,'hutom_id'].dropna().unique().tolist()
nums = [int(m.group(1)) for s in hutom_ids if (m := re.search(r'(\d+)$', s))]
id_number = np.sort(nums)[-1] + 1

sample_ids = sample_info['PatientID'].unique().tolist()
for idx in range(len(sample_ids)):
    try:
        dup = meta_all[meta_all['PatientID']==int(sample_ids[idx])]
    except:
        dup = meta_all[meta_all['PatientID']==sample_ids[idx]]
    if len(dup) == 0:
        hutomid = f"{organ[i]}{id_number:0{n_digits}d}"
        check_sample = sample_info[sample_info['PatientID'] == sample_ids[idx]]
        sample_info.loc[check_sample.index, "hutom_id"] = hutomid
        id_number += 1
    elif len(dup) > 0:
        hutomid = dup["hutom_id"].tolist()[0]
        check_sample = sample_info[sample_info['PatientID'] == sample_ids[idx]]
        sample_info.loc[check_sample.index, "hutom_id"] = hutomid
    else:
        print('check sample')
sample_info.loc[23,'hutom_id'] = 'UGI0407'
sample_info.loc[29,'hutom_id'] = 'UGI0416'

display(sample_info)


/disk1/users/da_cyh_0/miniconda3/envs/py3_11/lib/python3.11/site-packages/pydicom/valuerep.py:440: UserWarning: The value length (67) exceeds the maximum length of 64 allowed for VR UI. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)
/disk1/users/da_cyh_0/miniconda3/envs/py3_11/lib/python3.11/site-packages/pydicom/valuerep.py:440: UserWarning: The value length (65) exceeds the maximum length of 64 allowed for VR UI. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)


,hutom_id,PatientID,PatientName,PatientSex,PatientAge,PatientBirthDate,AcquisitionDate,PatientSize,PatientWeight,OtherPatientIDs,OtherPatientNames,InstitutionName,ReferringPhysicianName,AccessionNumber,Modality,BodyPartExamined,folder,Center,ImportDate
0,UGI3261,3885391,UNKNOWN,M,0Y,19000101,19000101,None,None,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,672050,CT,None,/nas/nas6/DataTeam/UGI/[CT]신촌세브란스 501/3885391/1,SEVERANCE,20251226
1,UGI3262,8120826,UNKNOWN,F,0Y,19000101,19000101,None,None,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,672080,CT,None,/nas/nas6/DataTeam/UGI/[CT]신촌세브란스 501/8120826/1,SEVERANCE,20251226
2,UGI0059,ANONYMIZE_0059,UNKNOWN,M,0Y,19000101,19000101,None,None,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,672037,CT,None,/nas/nas6/DataTeam/UGI/[CT]신촌세브란스 501/ANONYMIZ...,SEVERANCE,20251226
3,UGI0095,ANONYMIZE_0095,UNKNOWN,F,0Y,19000101,19000101,None,None,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,672070,CT,None,/nas/nas6/DataTeam/UGI/[CT]신촌세브란스 501/ANONYMIZ...,SEVERANCE,20251226
4,UGI0096,ANONYMIZE_0096,UNKNOWN,M,0Y,19000101,19000101,None,None,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,672069,CT,None,/nas/nas6/DataTeam/UGI/[CT]신촌세브란스 501/ANONYMIZ...,SEVERANCE,20251226
5,UGI0275,ANONYMIZE_0275,UNKNOWN,F,0Y,19000101,19000101,None,None,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,672159,CT,None,/nas/nas6/DataTeam/UGI/[CT]신촌세브란스 501/ANONYMIZ...,SEVERANCE,20251226
6,UGI0280,ANONYMIZE_0280,UNKNOWN,M,0Y,19000101,19000101,None,None,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,672163,CT,None,/nas/nas6/DataTeam/UGI/[CT]신촌세브란스 501/ANONYMIZ...,SEVERANCE,20251226
7,UGI0281,ANONYMIZE_0281,UNKNOWN,M,0Y,19000101,19000101,None,None,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,672164,CT,None,/nas/nas6/DataTeam/UGI/[CT]신촌세브란스 501/ANONYMIZ...,SEVERANCE,20251226
8,UGI0288,ANONYMIZE_0288,UNKNOWN,F,0Y,19000101,19000101,None,None,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,672171,CT,None,/nas/nas6/DataTeam/UGI/[CT]신촌세브란스 501/ANONYMIZ...,SEVERANCE,20251226
9,UGI0289,ANONYMIZE_0289,UNKNOWN,M,0Y,19000101,19000101,None,None,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,672172,CT,None,/nas/nas6/DataTeam/UGI/[CT]신촌세브란스 501/ANONYMIZ...,SEVERANCE,20251226


In [14]:
from tqdm.notebook import tqdm

anonymous_dir = os.path.join(dicom_dir[i], 'ANONYMOUS')
sample_ids = sample_info["PatientID"].unique().tolist()
for idx in tqdm(range(15, len(sample_ids))):
    # 4.1. 샘플 선택
    check_sample = sample_info[sample_info['PatientID'] == sample_ids[idx]]

    # 4.2. 폴더/경로, 영상 종류, HUTOM ID
    sample_path = check_sample["folder"].tolist()
    hutomid = check_sample['hutom_id'].tolist()    

    for m in range(len(sample_path)):
        # 샘플 기준, 하위 폴더 탐색
        posix = PurePath(sample_path[m])
        if len(posix.parts) == 8:
            dcm_path = PurePosixPath(*posix.parts[:-1])
        elif len(posix.parts) == 7:
            dcm_path = posix

        dcm_folder_list = check_dicom_from_folder(dcm_path, return_list=True)
        for n in range(len(dcm_folder_list)):
            # 하위 폴더 기준, 익명화: Name, ID > Hutom ID
            raw_dir = dcm_folder_list[n]
            save_dir = os.path.join(anonymous_dir, hutomid[m], PurePath(raw_dir).parts[-1])

            os.makedirs(save_dir, exist_ok=True)

            for dirpath, _, filenames in os.walk(raw_dir):
                for filename in filenames:
                    anonymize_dicom_file(raw_dir, filename, save_dir, id=hutomid[m])




  0%|          | 0/22 [00:00<?, ?it/s]

/disk1/users/da_cyh_0/miniconda3/envs/py3_11/lib/python3.11/site-packages/pydicom/valuerep.py:440: UserWarning: The value length (67) exceeds the maximum length of 64 allowed for VR UI. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)
/disk1/users/da_cyh_0/miniconda3/envs/py3_11/lib/python3.11/site-packages/pydicom/valuerep.py:440: UserWarning: The value length (65) exceeds the maximum length of 64 allowed for VR UI. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)


In [13]:
idx=15
check_sample = sample_info[sample_info['PatientID'] == sample_ids[idx]]
check_sample
# # 4.2. 폴더/경로, 영상 종류, HUTOM ID
# sample_path = check_sample["folder"].tolist()
# hutomid = check_sample['hutom_id'].tolist()    

# for m in range(len(sample_path)):
#     # 샘플 기준, 하위 폴더 탐색
#     posix = PurePath(sample_path[m])
#     dcm_path = PurePosixPath(*posix.parts[:-1])

# #     dcm_folder_list = check_dicom_from_folder(dcm_path, return_list=True)

#     break
# check_sample




,hutom_id,PatientID,PatientName,PatientSex,PatientAge,PatientBirthDate,AcquisitionDate,PatientSize,PatientWeight,OtherPatientIDs,OtherPatientNames,InstitutionName,ReferringPhysicianName,AccessionNumber,Modality,BodyPartExamined,folder,Center,ImportDate
15,UGI0383,ANONYMIZE_0383,UNKNOWN,M,0Y,19000101,19000101,None,None,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,672445,CT,None,/nas/nas6/DataTeam/UGI/[CT]신촌세브란스 501/ANONYMIZ...,SEVERANCE,20251226


In [36]:
from pydicom.dataset import FileMetaDataset, FileDataset
from pydicom.uid import ExplicitVRLittleEndian, generate_uid
from datetime import datetime

def ensure_valid_file_meta(ds: pydicom.dataset.Dataset):
    # 1) 본문에 섞인 0002 그룹 제거 (중요!)
    bad_0002 = [tag for tag in ds.keys() if tag.group == 0x0002]
    for tag in bad_0002:
        del ds[tag]

    # 2) file_meta를 FileMetaDataset로 강제
    fm = ds.file_meta if hasattr(ds, "file_meta") and isinstance(ds.file_meta, FileMetaDataset) else FileMetaDataset()

    # TransferSyntaxUID는 필수 (원본에 있으면 유지, 없으면 기본값 부여)
    if not getattr(fm, "TransferSyntaxUID", None):
        fm.TransferSyntaxUID = ExplicitVRLittleEndian

    # MediaStorage SOP UIDs는 있으면 맞춰줌
    if not getattr(fm, "MediaStorageSOPClassUID", None) and getattr(ds, "SOPClassUID", None):
        fm.MediaStorageSOPClassUID = ds.SOPClassUID

    if not getattr(fm, "MediaStorageSOPInstanceUID", None):
        fm.MediaStorageSOPInstanceUID = getattr(ds, "SOPInstanceUID", None) or generate_uid()

    if not getattr(fm, "ImplementationClassUID", None):
        fm.ImplementationClassUID = generate_uid()

    ds.file_meta = fm

    # preamble 보장 (없으면 추가)
    ds.preamble = getattr(ds, "preamble", b"\0" * 128)

    # TransferSyntaxUID에 맞춘 플래그 설정(기본: Explicit VR Little Endian)
    ds.is_little_endian = True
    ds.is_implicit_VR = False

    return ds

ds = ensure_valid_file_meta(ds)
ds.save_as(os.path.join(save_dir, filename))
